<div style='background:#0f172a;padding:28px 32px;border-radius:12px;margin-bottom:8px'>
<h1 style='color:white;margin:0 0 8px 0;font-size:22px'>EDA — Análisis de intensidad para segmentación clásica (BraTS 2024 GLI)</h1>
<h2 style='color:#94a3b8;margin:0 0 12px 0;font-size:15px;font-weight:400'>Secciones A–F · Complemento orientado a métodos clásicos con ITK (Otsu, K-means/GMM, crecimiento de regiones, watershed)</h2>
<div style='color:#64748b;font-size:13px'>Abel Albuez · Victoria Acero · Santiago Gil</div>
</div>

Este notebook complementa el EDA de la entrega 2 con el análisis de **dominio de intensidad**, que es lo que determina si los métodos clásicos van a funcionar. **No usa SimpleITK** (solo numpy/scipy/scikit-image/nibabel/plotly), así que corre directo en Colab sin el error de módulo.

**Cómo usarlo:** se puede correr de dos formas:
1. **Standalone (recomendado):** ejecuta *Run all*. Monta Drive, carga `df_vol` desde tu checkpoint `EDA_volumenes_completo.csv`, y **extrae del ZIP solo los ~25 casos de la muestra** (no los 2,200), así que es rápido.
2. **Anexado a tu EDA original:** pega estas celdas después de tus secciones S1–S11; reutiliza el `df_vol` que ya tengas en memoria.

Ajusta `DRIVE_DIR`, `DATASET_DIR` y `VOL_CSV` en la celda *Fuente de datos* si tus rutas difieren.

In [ ]:
# === Setup (sin SimpleITK) ===
# scikit-image, scipy, plotly y numpy ya vienen en Colab; solo aseguramos nibabel.
import importlib, subprocess, sys
for pkg, mod in [('nibabel','nibabel')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'])

import os, glob, gc, json, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nib
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats as spstats
from scipy import ndimage
from skimage.filters import threshold_multiotsu

print('Librerías cargadas (sin SimpleITK)')

In [ ]:
# === Fuente de datos (Drive) ===
import zipfile
try:
    from google.colab import drive
    if not os.path.ismount('/content/drive'):
        drive.mount('/content/drive')
    EN_COLAB = True
except Exception:
    EN_COLAB = False

# >>> AJUSTA ESTAS RUTAS SI HACE FALTA <<<
DRIVE_DIR   = '/content/drive/MyDrive/BRATS-2024'                    # carpeta con el ZIP y los checkpoints
DATASET_DIR = '/content/brats2024'                                  # dónde se extraen/buscan los NIfTI
VOL_CSV     = os.path.join(DRIVE_DIR, 'EDA_volumenes_completo.csv')  # checkpoint de volúmenes (entrega 2)

def _encontrar_zip(d):
    zips = sorted(glob.glob(os.path.join(d, '*.zip')))
    if not zips: return None
    return next((z for z in zips if 'TrainingData' in os.path.basename(z)), max(zips, key=os.path.getsize))

TRAINING_ZIP = _encontrar_zip(DRIVE_DIR)
print('Drive montado:', EN_COLAB)
print('ZIP del dataset:', TRAINING_ZIP)
print('Checkpoint de volúmenes existe:', os.path.exists(VOL_CSV))

In [ ]:
# === Constantes (se definen solo si no existen, para no pisar tu EDA original) ===
LABEL_MAP        = globals().get('LABEL_MAP', {1:'NETC', 2:'SNFH', 3:'ET', 4:'RC'})
MODALITIES_IMG   = ['t1n','t1c','t2w','t2f']                      # modalidades de imagen (sin seg)
MOD_LABELS       = {'t1n':'T1n','t1c':'T1c','t2w':'T2w','t2f':'T2-FLAIR'}
MOD_COLORS       = {'t1n':'#3B82F6','t1c':'#EF4444','t2w':'#22C55E','t2f':'#EAB308'}
SUBREGION_COLORS = {'ET':'#3B82F6','NETC':'#EF4444','SNFH':'#22C55E','RC':'#EAB308'}
SANO_COLOR       = '#94A3B8'

SAMPLE_N = 25      # casos para el análisis de intensidad (carga modalidades completas)
RNG_SEED = 42
MAXVOX_MOD  = 200_000   # voxeles por caso/modalidad para histogramas
MAXVOX_SUB  = 50_000    # voxeles por sub-región
MAXVOX_OTSU = 100_000   # voxeles para Otsu/bimodalidad

def _layout_oscuro(fig, titulo, h=400):
    fig.update_layout(title=dict(text=titulo, font=dict(size=14, color='white')),
        paper_bgcolor='#0f172a', plot_bgcolor='#1e293b',
        font=dict(color='white', size=11), height=h,
        legend=dict(bgcolor='#1e293b'))
    fig.update_xaxes(gridcolor='#334155'); fig.update_yaxes(gridcolor='#334155')
    return fig

print('Constantes listas. SAMPLE_N =', SAMPLE_N)

In [ ]:
# === Utilidades ===
def downsample(arr, maxn, seed=RNG_SEED):
    arr = np.asarray(arr).ravel()
    if arr.size <= maxn: return arr
    rng = np.random.default_rng(seed)
    return arr[rng.choice(arr.size, maxn, replace=False)]

def coef_bimodalidad(x):
    x = np.asarray(x); n = x.size
    if n < 4: return np.nan
    g1 = spstats.skew(x); g2 = spstats.kurtosis(x)   # kurtosis en exceso
    denom = g2 + 3.0*((n-1)**2)/((n-2)*(n-3))
    return (g1**2 + 1.0)/denom if denom != 0 else np.nan

def muestra_estratificada(df_vol, n=SAMPLE_N, seed=RNG_SEED):
    d = df_vol[df_vol['TumorTotal'] > 0].copy()
    if len(d) == 0: return df_vol['case_id'].head(n).tolist()
    try:
        d['q'] = pd.qcut(d['TumorTotal'], q=min(4, d['TumorTotal'].nunique()),
                         labels=False, duplicates='drop')
    except Exception:
        d['q'] = 0
    por_q = max(1, n // max(1, d['q'].nunique()))
    samp = d.groupby('q', group_keys=False).apply(
        lambda g: g.sample(min(len(g), por_q), random_state=seed))
    ids = samp['case_id'].tolist()
    for cid in d['case_id'].tolist():
        if len(ids) >= n: break
        if cid not in ids: ids.append(cid)
    return ids[:n]

def _dir_con_casos(base_dir, case_ids):
    if not os.path.isdir(base_dir): return None
    cset = set(case_ids)
    for root, dirs, _ in os.walk(base_dir):
        if cset & set(dirs): return root
    return None

def asegurar_casos(case_ids):
    # Garantiza en disco los NIfTI de case_ids. Usa DATASET_DIR si ya están;
    # si no, extrae SOLO esos casos del ZIP (rápido, no los 2,200).
    base = _dir_con_casos(DATASET_DIR, case_ids)
    faltan = [c for c in case_ids if not (base and os.path.isdir(os.path.join(base, c)))]
    if faltan:
        if not TRAINING_ZIP:
            raise FileNotFoundError(f'Sin datos en {DATASET_DIR} ni ZIP en {DRIVE_DIR}. Ajusta DRIVE_DIR/DATASET_DIR.')
        os.makedirs(DATASET_DIR, exist_ok=True)
        with zipfile.ZipFile(TRAINING_ZIP) as zf:
            miembros = zf.namelist()
            sel = [m for m in miembros if any((f'/{c}/' in m) or m.startswith(f'{c}/') for c in faltan)]
            for m in sel:
                zf.extract(m, DATASET_DIR)
        base = _dir_con_casos(DATASET_DIR, case_ids)
    if base is None:
        raise FileNotFoundError('No se localizaron los casos tras la extracción; revisa el contenido del ZIP.')
    return base

def _ruta(case_id, mod):
    row = df_ok[df_ok['case_id'] == case_id]
    if row.empty: return None
    return row.iloc[0].get(f'{mod}_path')

def cargar_modalidad(case_id, mod):
    return nib.load(_ruta(case_id, mod)).get_fdata().astype(np.float32)

def cargar_seg(case_id):
    return nib.load(_ruta(case_id, 'seg')).get_fdata().astype(np.int16)

print('Utilidades listas.')

In [ ]:
# === df_vol + muestra + extracción de SOLO la muestra + df_ok ===
# 1) df_vol: reutiliza el de memoria o cárgalo del checkpoint CSV (no toca imágenes).
if 'df_vol' not in globals() or df_vol is None:
    if os.path.exists(VOL_CSV):
        df_vol = pd.read_csv(VOL_CSV)
        print('df_vol cargado del checkpoint:', len(df_vol), 'casos')
    else:
        raise FileNotFoundError(
            'No hay df_vol en memoria ni checkpoint en ' + VOL_CSV + '.\n'
            'Opciones: (a) ejecuta estas celdas anexadas a tu EDA original (que crea df_vol), '
            'o (b) coloca EDA_volumenes_completo.csv en DRIVE_DIR.')
else:
    print('df_vol en memoria:', len(df_vol), 'casos')

# 2) Muestra estratificada y extracción SOLO de esos casos (rápido).
SAMPLE_IDS = muestra_estratificada(df_vol)
print(f'Muestra: {len(SAMPLE_IDS)} casos -> asegurando sus NIfTI...')
CASES_BASE = asegurar_casos(SAMPLE_IDS)
print('Casos disponibles en:', CASES_BASE)

# 3) df_ok solo para la muestra (rutas a cada modalidad + seg).
_recs = []
for cid in SAMPLE_IDS:
    cp = os.path.join(CASES_BASE, cid)
    row = {'case_id': cid}
    for mod in MODALITIES_IMG + ['seg']:
        hits = glob.glob(os.path.join(cp, f'*-{mod}.nii*'))
        row[f'{mod}_path'] = hits[0] if hits else None
    _recs.append(row)
df_ok = pd.DataFrame(_recs)
_faltan = df_ok[[f'{m}_path' for m in MODALITIES_IMG + ['seg']]].isna().any(axis=1).sum()
print('df_ok (muestra):', len(df_ok), 'casos |', _faltan, 'con archivos faltantes')

In [ ]:
# === Pasada única de recolección sobre la muestra ===
# Carga cada modalidad y la seg una sola vez por caso y extrae todo lo que A-E necesitan.
intens_mod  = {m: [] for m in MODALITIES_IMG}                                  # A
intens_sub  = {m: {ln: [] for ln in LABEL_MAP.values()} for m in MODALITIES_IMG}  # B
intens_sano = {m: [] for m in MODALITIES_IMG}                                  # B
stats_caso  = []                                                               # D
otsu_bimod  = []                                                               # C
semillas    = []                                                               # E

for k, cid in enumerate(SAMPLE_IDS, 1):
    try:
        seg = cargar_seg(cid)
    except Exception as e:
        print('  seg falló', cid, e); continue

    # Centroide del tumor (ET; si no hay, todo el tumor) para análisis de semillas
    mask_et = (seg == 3)
    if mask_et.sum() > 0:
        c = ndimage.center_of_mass(mask_et)
    elif (seg > 0).sum() > 0:
        c = ndimage.center_of_mass(seg > 0)
    else:
        c = None
    ci = tuple(int(round(v)) for v in c) if c is not None else None

    for m in MODALITIES_IMG:
        try:
            vol = cargar_modalidad(cid, m)
        except Exception:
            continue
        bm = vol > 0                 # máscara de cerebro (BraTS: fondo = 0)
        bv = vol[bm]
        if bv.size == 0:
            del vol; continue

        intens_mod[m].append(downsample(bv, MAXVOX_MOD))                       # A
        stats_caso.append({'case_id': cid, 'modalidad': m,                     # D
                           'min': float(bv.min()), 'max': float(bv.max()),
                           'media': float(bv.mean()), 'p99': float(np.percentile(bv, 99))})

        for lv, ln in LABEL_MAP.items():                                       # B sub-regiones
            sv = vol[seg == lv]
            if sv.size:
                intens_sub[m][ln].append(downsample(sv, MAXVOX_SUB))
        sano = vol[bm & (seg == 0)]                                            # B tejido sano
        if sano.size:
            intens_sano[m].append(downsample(sano, MAXVOX_MOD))

        bvo = downsample(bv, MAXVOX_OTSU)                                       # C Otsu + bimodalidad
        rec = {'case_id': cid, 'modalidad': m, 'bimodalidad': coef_bimodalidad(bvo)}
        try:
            rec['otsu_n2_t1'] = float(threshold_multiotsu(bvo, classes=2)[0])
        except Exception:
            rec['otsu_n2_t1'] = np.nan
        try:
            t3 = threshold_multiotsu(bvo, classes=3)
            rec['otsu_n3_t1'], rec['otsu_n3_t2'] = float(t3[0]), float(t3[1])
        except Exception:
            rec['otsu_n3_t1'] = rec['otsu_n3_t2'] = np.nan
        otsu_bimod.append(rec)

        if m == 't1c' and ci is not None:                                      # E semilla en T1c
            z, y, x = ci
            z = min(max(z,0), vol.shape[0]-1); y = min(max(y,0), vol.shape[1]-1); x = min(max(x,0), vol.shape[2]-1)
            vecindad = vol[max(0,z-1):z+2, max(0,y-1):y+2, max(0,x-1):x+2]
            semillas.append({'case_id': cid, 'a0': z, 'a1': y, 'a2': x,
                             'intensidad_t1c': float(vol[z, y, x]),
                             'media_vecindad': float(vecindad.mean())})
        del vol, bv
    del seg; gc.collect()
    if k % 5 == 0 or k == len(SAMPLE_IDS):
        print(f'  {k}/{len(SAMPLE_IDS)} casos procesados')

# Consolidar acumuladores
intens_mod  = {m: (np.concatenate(v) if v else np.array([])) for m, v in intens_mod.items()}
intens_sano = {m: (np.concatenate(v) if v else np.array([])) for m, v in intens_sano.items()}
for m in MODALITIES_IMG:
    for ln in LABEL_MAP.values():
        intens_sub[m][ln] = np.concatenate(intens_sub[m][ln]) if intens_sub[m][ln] else np.array([])
df_stats  = pd.DataFrame(stats_caso)
df_otsu   = pd.DataFrame(otsu_bimod)
df_seed   = pd.DataFrame(semillas)
print('Recolección completa.')

---
## A — Histogramas de intensidad por modalidad

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Distribución de intensidad del cerebro (voxeles &gt; 0, fondo enmascarado) por modalidad. Es la base de Otsu, umbral y K-means: la forma uni/bimodal del histograma anticipa si un umbral global puede separar tejido tumoral del sano.
</div>

In [ ]:
figA = go.Figure()
for m in MODALITIES_IMG:
    v = intens_mod[m]
    if v.size == 0: continue
    lo, hi = np.percentile(v, [0.5, 99.5])
    figA.add_trace(go.Histogram(x=v, nbinsx=120, name=MOD_LABELS[m],
        marker_color=MOD_COLORS[m], opacity=0.6,
        xbins=dict(start=float(lo), end=float(hi), size=(hi-lo)/120)))
figA.update_layout(barmode='overlay', xaxis_title='Intensidad', yaxis_title='Frecuencia')
_layout_oscuro(figA, f'Histograma de intensidad por modalidad — muestra de {len(SAMPLE_IDS)} casos')
figA.show()

---
## B — Separabilidad de intensidades: sub-región vs tejido sano

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Para cada modalidad se comparan las intensidades de cada sub-región (ET, NETC, SNFH, RC) contra el parénquima sano. La <b>separabilidad</b> se cuantifica con el coeficiente de solapamiento de histogramas (OVL: 0 = no se solapan, 1 = idénticos) y la distancia de Bhattacharyya (mayor = más separable). Esto justifica qué método/umbral conviene por sub-región y modalidad.
</div>

In [ ]:
def ovl_bhatt(a, b, bins=128):
    if a.size == 0 or b.size == 0: return np.nan, np.nan
    lo = min(a.min(), b.min()); hi = max(a.max(), b.max())
    if hi <= lo: return np.nan, np.nan
    ha, _ = np.histogram(a, bins=bins, range=(lo, hi), density=True)
    hb, _ = np.histogram(b, bins=bins, range=(lo, hi), density=True)
    w = (hi-lo)/bins
    pa, pb = ha*w, hb*w
    ovl = float(np.minimum(pa, pb).sum())
    bc  = float(np.sqrt(pa*pb).sum())
    bdist = float(-np.log(bc)) if bc > 0 else np.inf
    return ovl, bdist

# Violines por modalidad
figB = make_subplots(rows=2, cols=2, subplot_titles=[MOD_LABELS[m] for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i//2 + 1, i % 2 + 1
    sano = intens_sano[m]
    if sano.size:
        figB.add_trace(go.Violin(y=downsample(sano, 8000), name='Sano',
            line_color=SANO_COLOR, showlegend=(i == 0), box_visible=True, meanline_visible=True),
            row=r, col=c)
    for ln in ['ET','NETC','SNFH','RC']:
        v = intens_sub[m][ln]
        if v.size:
            figB.add_trace(go.Violin(y=downsample(v, 8000), name=ln,
                line_color=SUBREGION_COLORS[ln], showlegend=(i == 0),
                box_visible=True, meanline_visible=True), row=r, col=c)
_layout_oscuro(figB, 'Distribución de intensidad por clase y modalidad', h=720)
figB.show()

# Tablas de separabilidad (OVL y Bhattacharyya)
ovl_tab, bh_tab = {}, {}
for m in MODALITIES_IMG:
    ovl_tab[MOD_LABELS[m]] = {}
    bh_tab[MOD_LABELS[m]]  = {}
    for ln in ['ET','NETC','SNFH','RC']:
        o, b = ovl_bhatt(intens_sub[m][ln], intens_sano[m])
        ovl_tab[MOD_LABELS[m]][ln] = round(o, 3) if o == o else np.nan
        bh_tab[MOD_LABELS[m]][ln]  = round(b, 3) if b == b else np.nan
df_ovl = pd.DataFrame(ovl_tab)
df_bh  = pd.DataFrame(bh_tab)
print('Solapamiento (OVL) sub-región vs sano  —  menor = más separable')
display(df_ovl.style.background_gradient(cmap='RdYlGn_r', axis=None)
        .set_caption('OVL (0=separable, 1=indistinguible)'))
print('\nDistancia de Bhattacharyya  —  mayor = más separable')
display(df_bh.style.background_gradient(cmap='RdYlGn', axis=None)
        .set_caption('Bhattacharyya'))

---
## C — Bimodalidad y vista previa del umbral de Otsu

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Otsu asume un histograma con modas separables. Se calcula el coeficiente de bimodalidad (regla práctica: &gt; 0.555 sugiere bimodalidad) y los umbrales de Otsu para 2 y 3 clases por modalidad. Predice dónde Otsu separará bien (bimodal) y dónde fallará (unimodal).
</div>

In [ ]:
resumen_C = df_otsu.groupby('modalidad').agg(
    bimodalidad=('bimodalidad','median'),
    otsu_n2_t1=('otsu_n2_t1','median'),
    otsu_n3_t1=('otsu_n3_t1','median'),
    otsu_n3_t2=('otsu_n3_t2','median'),
).round(3)
resumen_C['bimodal?'] = resumen_C['bimodalidad'] > 0.555
resumen_C = resumen_C.reindex(MODALITIES_IMG)
resumen_C.index = [MOD_LABELS[m] for m in resumen_C.index]
display(resumen_C.style.set_caption('Bimodalidad y umbrales de Otsu (mediana de la muestra)'))

# Histogramas con líneas de umbral Otsu (mediana) por modalidad
figC = make_subplots(rows=2, cols=2, subplot_titles=[MOD_LABELS[m] for m in MODALITIES_IMG])
for i, m in enumerate(MODALITIES_IMG):
    r, c = i//2 + 1, i % 2 + 1
    v = intens_mod[m]
    if v.size == 0: continue
    lo, hi = np.percentile(v, [0.5, 99.5])
    figC.add_trace(go.Histogram(x=v, nbinsx=100, marker_color=MOD_COLORS[m],
        opacity=0.7, showlegend=False,
        xbins=dict(start=float(lo), end=float(hi), size=(hi-lo)/100)), row=r, col=c)
    med = df_otsu[df_otsu['modalidad'] == m].median(numeric_only=True)
    for col, dash in [('otsu_n3_t1','dash'), ('otsu_n3_t2','dash')]:
        th = med.get(col, np.nan)
        if th == th:
            figC.add_vline(x=float(th), line=dict(color='white', width=1.5, dash=dash), row=r, col=c)
_layout_oscuro(figC, 'Histograma con umbrales de Otsu (n=3, mediana)', h=720)
figC.show()

---
## D — Variabilidad del rango de intensidades entre casos

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Las intensidades de MRI no están en unidades estándar: varían entre casos e instituciones. Si la dispersión de la media/p99 entre casos es alta, un umbral fijo no es transferible y se necesita <b>normalización</b> (z-score o histogram matching) como paso de limpieza antes de segmentar.
</div>

In [ ]:
figD = make_subplots(rows=1, cols=2, subplot_titles=['Media por caso', 'p99 por caso'])
for m in MODALITIES_IMG:
    sub = df_stats[df_stats['modalidad'] == m]
    figD.add_trace(go.Box(y=sub['media'], name=MOD_LABELS[m], marker_color=MOD_COLORS[m],
        showlegend=False, boxmean=True), row=1, col=1)
    figD.add_trace(go.Box(y=sub['p99'], name=MOD_LABELS[m], marker_color=MOD_COLORS[m],
        showlegend=False, boxmean=True), row=1, col=2)
_layout_oscuro(figD, 'Dispersión de intensidad entre casos (motiva normalización)')
figD.show()

tab_var = df_stats.groupby('modalidad')[['media','p99']].agg(['mean','std']).round(2)
tab_var['CV_media_%'] = (df_stats.groupby('modalidad')['media'].std() /
                         df_stats.groupby('modalidad')['media'].mean() * 100).round(1)
tab_var.index = [MOD_LABELS[m] for m in tab_var.index]
display(tab_var)

---
## E — Análisis de semillas para crecimiento de regiones

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
Para <code>ConnectedThreshold</code> / <code>ConfidenceConnected</code> hay que elegir una semilla y un rango de tolerancia. Se mide la intensidad en T1c en el centroide del tumor (ET) entre casos. El rango p10–p90 sugiere ventanas razonables de intensidad de semilla, y la dispersión orienta la tolerancia.
</div>

In [ ]:
if len(df_seed) > 0:
    s = df_seed['intensidad_t1c']
    p10, p50, p90 = np.percentile(s, [10, 50, 90])
    figE = go.Figure()
    figE.add_trace(go.Histogram(x=s, nbinsx=25, marker_color='#EF4444', opacity=0.8,
        name='Intensidad en centroide (T1c)'))
    for x, lbl in [(p10,'p10'), (p50,'mediana'), (p90,'p90')]:
        figE.add_vline(x=float(x), line=dict(color='white', dash='dash'),
                       annotation_text=lbl, annotation_font_color='white')
    figE.update_layout(xaxis_title='Intensidad T1c en el centroide del tumor', yaxis_title='Casos')
    _layout_oscuro(figE, 'Intensidad de semilla candidata (T1c) entre casos')
    figE.show()
    tol = float(df_seed['intensidad_t1c'].std())
    print(f'Rango de intensidad de semilla (T1c): p10={p10:.1f}  mediana={p50:.1f}  p90={p90:.1f}')
    print(f'Tolerancia sugerida para ConnectedThreshold (~1 sigma): +/- {tol:.1f}')
    display(df_seed.round(1).head(15))
else:
    print('Sin datos de semilla (no se encontró ET ni tumor en la muestra).')

---
## F — Subconjunto representativo de casos para el prototipo

<div style='background:#f8fafc;border-left:4px solid #8B5CF6;padding:14px 18px;border-radius:0 8px 8px 0;margin:12px 0'>
El prototipo clásico no corre sobre los ~2,200 casos: se eligen 4–6 casos demostrativos que cubran los retos. Se exporta <code>casos_demostrativos.csv</code> para que <b>todo el equipo segmente y visualice los mismos casos</b>.
</div>

In [ ]:
dv = df_vol[df_vol['TumorTotal'] > 0].copy()
sel = {}
def _pick(mask_series, etiqueta, criterio):
    cand = dv[mask_series]
    if len(cand) == 0: return
    cid = criterio(cand)
    sel.setdefault(cid, []).append(etiqueta)

# típico (mediana de TumorTotal)
med = dv['TumorTotal'].median()
_pick(pd.Series(True, index=dv.index), 'Típico (mediana de tumor)',
      lambda c: c.iloc[(c['TumorTotal']-med).abs().argsort()].iloc[0]['case_id'])
# grande (p95) y pequeño (p5)
_pick(pd.Series(True, index=dv.index), 'Tumor grande (~p95)',
      lambda c: c.sort_values('TumorTotal').iloc[int(len(c)*0.95)-1]['case_id'])
_pick(pd.Series(True, index=dv.index), 'Tumor pequeño (~p5)',
      lambda c: c.sort_values('TumorTotal').iloc[int(len(c)*0.05)]['case_id'])
# post-resección (RC > 0)
_pick(dv['RC'] > 0, 'Post-resección (RC>0)',
      lambda c: c.sort_values('RC', ascending=False).iloc[0]['case_id'])
# solo edema (ET == 0, SNFH > 0)
_pick((dv['ET'] == 0) & (dv['SNFH'] > 0), 'Solo edema (sin ET)',
      lambda c: c.sort_values('SNFH', ascending=False).iloc[0]['case_id'])
# NETC dominante
_pick(dv['NETC'] > 0, 'NETC dominante',
      lambda c: c.assign(r=c['NETC']/c['TumorTotal']).sort_values('r', ascending=False).iloc[0]['case_id'])

rows = []
for cid, etiquetas in sel.items():
    r = df_vol[df_vol['case_id'] == cid].iloc[0]
    rows.append({'case_id': cid, 'motivo': ' / '.join(etiquetas),
                 'TumorTotal_cc': r['TumorTotal'],
                 **{ln: r[ln] for ln in LABEL_MAP.values()}})
df_demo = pd.DataFrame(rows)
out_csv = 'casos_demostrativos.csv'
df_demo.to_csv(out_csv, index=False)
print('Casos demostrativos guardados en', os.path.abspath(out_csv))
display(df_demo)

---
## Nota sobre S6 (parches) y S10 (pesos de pérdida) de la entrega 2

Esas dos secciones del EDA original están orientadas a **aprendizaje profundo** (muestreo por parches y pesos de clase para la función de pérdida), enfoque que **no aplica a este proyecto** (segmentación clásica con ITK). Su contenido se reinterpreta así:

- **Desbalance de clases (S6/S10) → prioridad de sub-región para métodos clásicos.** Las sub-regiones minoritarias (ET, NETC) son las más difíciles de aislar por intensidad; las secciones B y C indican con qué modalidad y umbral abordarlas.
- **No se reportan pesos de pérdida ni diseño de red.** El "diseño" aquí es la elección del método clásico por sub-región (umbral/Otsu para ET en T1c, crecimiento de regiones con la semilla de la sección E, watershed/K-means para regiones difusas).

---
## Reporte HTML autocontenido (A–F)

Genera un HTML con todas las figuras y tablas de este complemento, en el mismo estilo oscuro. Es el entregable clave de la tarea 1.

In [ ]:
import plotly.io as pio

def fig_html(fig, incluir_js):
    return pio.to_html(fig, full_html=False,
                       include_plotlyjs=('inline' if incluir_js else False),
                       default_width='100%', default_height='460px')

def tabla_html(df, titulo):
    return (f"<div class='stitle'>{titulo}</div>" +
            df.to_html(border=0, classes='tbl', float_format=lambda x: f'{x:.3f}'))

secciones = [
    ('A — Histograma de intensidad por modalidad', [figA], []),
    ('B — Separabilidad sub-región vs sano', [figB],
        [('Solapamiento (OVL) — menor = más separable', df_ovl),
         ('Distancia de Bhattacharyya — mayor = más separable', df_bh)]),
    ('C — Bimodalidad y umbrales de Otsu', [figC],
        [('Bimodalidad y Otsu (mediana)', resumen_C)]),
    ('D — Variabilidad de intensidad entre casos', [figD], [('Resumen de variabilidad', tab_var)]),
]
if len(df_seed) > 0:
    secciones.append(('E — Semillas para crecimiento de regiones', [figE], []))
secciones.append(('F — Subconjunto representativo', [], [('Casos demostrativos', df_demo)]))

partes, primero = [], True
for titulo, figs, tablas in secciones:
    partes.append(f"<section><h2>{titulo}</h2>")
    for f in figs:
        partes.append(fig_html(f, primero)); primero = False
    for tt, df in tablas:
        partes.append(tabla_html(df, tt))
    partes.append("</section>")

nav = "".join(f"<a href='#s{i}'>{s[0].split(' — ')[0]}</a>" for i, s in enumerate(secciones))
secciones_html = "".join(
    p.replace("<section>", f"<section id='s{i}'>", 1) if p.startswith("<section>") else p
    for i, p in enumerate(partes))

html = f"""<!doctype html><html lang='es'><head><meta charset='utf-8'>
<title>EDA intensidad — BraTS 2024 GLI (segmentación clásica)</title>
<style>
 body{{background:#0f172a;color:#e2e8f0;font-family:system-ui,Arial;margin:0;padding:0 0 60px}}
 header{{background:#0f172a;padding:28px 32px;border-bottom:1px solid #1e293b}}
 header h1{{margin:0;font-size:22px}} header p{{color:#94a3b8;margin:6px 0 0}}
 nav{{position:sticky;top:0;background:#0f172a;padding:10px 32px;border-bottom:1px solid #1e293b;z-index:5}}
 nav a{{color:#60a5fa;margin-right:14px;text-decoration:none;font-size:13px}}
 section{{padding:22px 32px;border-bottom:1px solid #1e293b}}
 section h2{{font-size:17px;color:#fff}}
 .stitle{{color:#94a3b8;font-size:13px;margin:14px 0 6px}}
 table.tbl{{border-collapse:collapse;font-size:12px;margin:4px 0 14px}}
 table.tbl th{{background:#1e293b;color:#fff;padding:7px 12px;text-align:left}}
 table.tbl td{{padding:6px 12px;border-top:1px solid #1e293b;color:#cbd5e1}}
</style></head><body>
<header><h1>EDA — Análisis de intensidad para segmentación clásica</h1>
<p>BraTS 2024 GLI · muestra de {len(SAMPLE_IDS)} casos · Abel Albuez, Victoria Acero, Santiago Gil</p></header>
<nav>{nav}</nav>
{secciones_html}
</body></html>"""

out_html = 'EDA_BraTS2024_GLI_reporte_clasico.html'
with open(out_html, 'w', encoding='utf-8') as f:
    f.write(html)
print('Reporte HTML guardado en', os.path.abspath(out_html))

# Descarga automática en Colab
try:
    from google.colab import files
    files.download(out_html)
except Exception:
    pass